# NMR metabolite correlation

Pairwise Pearson correlation of the 250 NMR metabolites in `data/assay.csv`, the assay matrix used for the incident and complication models. Metabolites are ordered by average-linkage clustering of $1 - r$.


In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(here)
})


In [ ]:
assay <- read.csv(here("data", "assay.csv"), row.names = 1)
nmr_names <- setdiff(names(assay), "StudyID")

stopifnot(length(nmr_names) == 250L)
stopifnot(!anyDuplicated(assay$StudyID))


# Correlation matrix

Pairwise Pearson *r* across the 250 NMR metabolites. Columns follow the metabolite order in `assay.csv`.


In [ ]:
nmr_mat <- assay %>%
  dplyr::select(dplyr::all_of(nmr_names)) %>%
  as.matrix()
storage.mode(nmr_mat) <- "double"

cat("samples x metabolites:", nrow(nmr_mat), "x", ncol(nmr_mat), "\n")
cat("NA count:", sum(is.na(nmr_mat)), "\n")

nmr_cor <- cor(nmr_mat, method = "pearson", use = "pairwise.complete.obs")
stopifnot(identical(dim(nmr_cor), c(length(nmr_names), length(nmr_names))))
dimnames(nmr_cor) <- list(nmr_names, nmr_names)


In [ ]:
r_off <- nmr_cor[upper.tri(nmr_cor)]

cat("n pairs:", length(r_off), "\n")
print(summary(r_off))
cat(
  sprintf(
    "|r| >= 0.8: %s (%.1f%%)\n|r| >= 0.9: %s (%.1f%%)\n",
    sum(abs(r_off) >= 0.8), 100 * mean(abs(r_off) >= 0.8),
    sum(abs(r_off) >= 0.9), 100 * mean(abs(r_off) >= 0.9)
  )
)


In [ ]:
hc <- hclust(as.dist(1 - nmr_cor), method = "average")
ord <- hc$order
nmr_cor_ord <- nmr_cor[ord, ord]

nmr_cor_long <- nmr_cor_ord %>%
  as.data.frame() %>%
  tibble::rownames_to_column("metabolite_x") %>%
  tidyr::pivot_longer(-metabolite_x, names_to = "metabolite_y", values_to = "r") %>%
  dplyr::mutate(
    metabolite_x = factor(metabolite_x, levels = colnames(nmr_cor_ord)),
    metabolite_y = factor(metabolite_y, levels = colnames(nmr_cor_ord))
  )

p_nmr_cor <- ggplot(nmr_cor_long, aes(x = metabolite_x, y = metabolite_y, fill = r)) +
  geom_raster() +
  scale_fill_gradient2(
    low = "steelblue", mid = "white", high = "firebrick",
    midpoint = 0, limits = c(-1, 1), name = "Pearson r"
  ) +
  coord_fixed() +
  labs(
    title = "Pairwise Pearson correlation of 250 NMR metabolites",
    x = "",
    y = ""
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 3.5),
    axis.text.y = element_text(size = 3.5),
    axis.title = element_text(size = 10),
    axis.ticks = element_line(linewidth = 0.2),
    panel.grid = element_blank(),
    plot.title = element_text(size = 12)
  )

p_nmr_cor

FIG_DIR <- here("outputs/figures")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)
ggsave(
  file.path(FIG_DIR, "NMR_CORR.png"),
  plot = p_nmr_cor,
  width = 10,
  height = 10,
  dpi = 600
)
